# import data

In [19]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import KFold, GroupKFold, cross_val_predict, train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

from clean_zenodo import bersihkan

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
plt.rcParams["figure.figsize"] = (8, 4)

PATH_DATA = "../data/dataset_raw_29747.csv"
PATH_MODEL = "../models/model_v2.joblib"

In [20]:
df, laporan = bersihkan(PATH_DATA)
for k, v in laporan.items():
    print(f"{k:35s} {v:>7,}")
print("\nUkuran data bersih:", df.shape)
df.head()

baris_awal                           29,747
setelah_buang_kosong_dan_header      29,569
setelah_harga_tahun_valid            28,797
setelah_hapus_url_ganda              25,494
setelah_hapus_iklan_kembar           18,885
baris_akhir                          18,885

Ukuran data bersih: (18885, 15)


,sumber,batch,merek,model,varian,tahun,km,transmisi,bahan_bakar,mesin_l,bodi,kota,tgl,bulan_idx,harga
0,OLX,B1-2025,Mercedes-Benz,A-Class,2.0 Bensin-AT,2014,NaN,Automatic,Bensin,2.0,Hatchback,Jakarta Timur,2025-10-07,10.0,"499,000,000.0"
1,OLX,B1-2025,Mercedes-Benz,Sprinter,NaN,2021,NaN,Manual,Diesel,NaN,Tidak diketahui,Jakarta Utara,2025-10-17,10.0,"1,850,000,000.0"
2,OLX,B1-2025,Mercedes-Benz,GLC-Class,2.0 AMG Line Bensin-AT,2022,NaN,Automatic,Bensin,2.0,SUV,Jakarta Barat,2025-10-06,10.0,"720,000,000.0"
3,OLX,B1-2025,Mercedes-Benz,GLE-Class,3.0 Coupe AMG Line Bensin-AT,2016,NaN,Automatic,Bensin,3.0,SUV,Jakarta Selatan,2025-10-10,10.0,"598,000,000.0"
4,OLX,B1-2025,Mercedes-Benz,Lainnya,Lain-lain-AT,2010,NaN,Automatic,Bensin,NaN,Tidak diketahui,Denpasar Kota,2025-10-23,10.0,"159,000,000.0"


# EDA

In [21]:
print("Merek:")
print(df["merek"].value_counts().to_string())
print("\nKelengkapan kolom (bagian yang terisi):")
print(df[["km", "mesin_l", "transmisi", "bahan_bakar", "bodi"]].notna().mean().round(3).to_string())
print("\nKilometer terisi per batch:")
print(df.groupby("batch")["km"].apply(lambda s: s.notna().mean()).round(3).to_string())

Merek:
merek
Honda            4749
Toyota           3640
Daihatsu         3498
Mercedes-Benz    2998
Suzuki           2914
Mitsubishi        439
Wuling            409
BYD               226
Denza               4
Hyundai             2
BMW                 1
Chery               1
MINI                1
Mazda               1
DFSK                1
Smart               1

Kelengkapan kolom (bagian yang terisi):
km            0.5
mesin_l       1.0
transmisi     1.0
bahan_bakar   1.0
bodi          1.0

Kilometer terisi per batch:
batch
B1-2025   0.8
B2-2026   0.2
